### Broadcast Join in PySpark
- Broadcast Join is a join optimization technique that improves performance by sending a small DataFrame to all executors.
- Avoids shuffling large amounts of data across the cluster during joins.
- `spark.sql.autoBroadcastJoinThreshold` controls the size limit for automatic broadcast consideration.
- `explain()` helps inspect the physical execution plan and identify the join strategy.
- Broadcast joins can be triggered using `hint("broadcast")` in PySpark or `BROADCAST` hints in Spark SQL.
- Useful when joining a large DataFrame with a sufficiently small DataFrame that can fit in executor memory.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import Row

# Sample Employee info
data_employee = [
    Row(employee_id=1, name='Alice', city='New York'),
    Row(employee_id=2, name='Bob', city='Los Angeles'),
    Row(employee_id=3, name='Charlie', city='Chicago'),
    Row(employee_id=4, name='David', city='Houston'),
    Row(employee_id=5, name='Eva', city='Phoenix')
]

# Sample Salary info
data_salary = [
    Row(employee_id=1, salary=70000, deprt='HR'),
    Row(employee_id=2, salary=80000, deprt='Engineering'),
    Row(employee_id=3, salary=65000, deprt='Finance'),
    Row(employee_id=6, salary=90000, deprt='Marketing'),
    Row(employee_id=7, salary=75000, deprt='Sales')
]

spark = SparkSession.getActiveSession()

df_employee = spark.createDataFrame(data_employee)
df_salary = spark.createDataFrame(data_salary)

# Register temp views for SQL
df_employee.createOrReplaceTempView('employee')
df_salary.createOrReplaceTempView('salary')

display(df_employee)
display(df_salary)

### Broadcast Join Threshold

- `spark.sql.autoBroadcastJoinThreshold` defines the maximum size of a table that Spark can automatically broadcast during a join.
- Default is typically 10 MB.
- If a table is smaller than the threshold, Spark may choose a Broadcast Join to improve performance.

In [0]:
threshold = spark.conf.get("spark.sql.autoBroadcastJoinThreshold", "Configuration not available")
print(threshold)

### Setting Broadcast Join Threshold

- The broadcast join threshold can be changed using `spark.conf.set()`.
- Increasing the threshold allows Spark to consider broadcasting larger tables.
- Broadcasting can reduce shuffle during joins, but broadcasting large tables can increase memory usage.

In [0]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "1gb")

threshold = spark.conf.get("spark.sql.autoBroadcastJoinThreshold")
print(threshold)

### Getting Physical Plan

- `explain()` displays the query execution plan of a DataFrame.
- It helps understand how Spark executes operations such as joins.
- The physical plan can show strategies such as:
  - Broadcast Hash Join
  - Sort Merge Join
  - Shuffled Hash Join

In [0]:
df_employee.join(
    df_salary,
    on="employee_id",
    how="inner"
).explain()

In [0]:
%sql

EXPLAIN
SELECT *
FROM employee
INNER JOIN salary
ON employee.employee_id = salary.employee_id;

### Hint for Broadcast Join

- A broadcast hint tells Spark to broadcast a specified DataFrame or table during a join.
- It can help avoid shuffling the larger DataFrame.
- Broadcast hints can influence Spark's join strategy even when the table exceeds the automatic broadcast threshold.
- Use broadcast joins when the selected table is small enough to fit comfortably in executor memory.

In [0]:
df = df_employee.join(
    df_salary.hint("broadcast"), #.hint(shuffle_hash) for shuffle hash join
    on="employee_id",
    how="inner"
)

display(df)